# Bronze Layer — ingestão dos dados brutos

Neste notebook vamos ler os oito arquivos enviados para o Volume e criar uma tabela Bronze para cada um deles. Nesta etapa não faremos tratamentos nos dados: as colunas serão mantidas como texto para que os ajustes sejam feitos na Silver.

Como mais de uma pessoa do grupo pode executar o notebook, usamos `overwrite` para não acumular os mesmos registros mais de uma vez.

In [0]:
catalog = "v_credit"
bronze_schema = f"{catalog}.bronze"
landing_path = f"/Volumes/{catalog}/bronze/landing"

## Caminhos dos arquivos

In [0]:
path_base_atendentes = f"{landing_path}/base_atendentes.csv"
path_base_motivos = f"{landing_path}/base_motivos.csv"
path_canais = f"{landing_path}/canais.csv"
path_chamados = f"{landing_path}/chamados.csv"
path_chamados_hora = f"{landing_path}/Chamados_Hora.CSV"
path_clientes = f"{landing_path}/clientes.csv"
path_custos = f"{landing_path}/custos.csv"
path_pesquisa_satisfacao = f"{landing_path}/pesquisa_satisfacao.csv"

## Leitura dos arquivos

A leitura é feita sem `inferSchema`, então as colunas permanecem como texto nesta camada. Nos arquivos `Chamados_Hora.CSV` e `clientes.csv`, informamos a codificação `windows-1252` para manter os caracteres acentuados.

In [0]:
df_base_atendentes_raw = spark.read.csv(path_base_atendentes, header=True)
df_base_motivos_raw = spark.read.csv(path_base_motivos, header=True)
df_canais_raw = spark.read.csv(path_canais, header=True)
df_chamados_raw = spark.read.csv(path_chamados, header=True)
df_chamados_hora_raw = spark.read.option("encoding", "windows-1252").csv(path_chamados_hora, header=True)
df_clientes_raw = spark.read.option("encoding", "windows-1252").csv(path_clientes, header=True)
df_custos_raw = spark.read.csv(path_custos, header=True)
df_pesquisa_satisfacao_raw = spark.read.csv(path_pesquisa_satisfacao, header=True)

## Gravação das tabelas Bronze

As tabelas são gravadas com `overwrite`. Dessa forma, se outro integrante executar o notebook, os dados serão carregados novamente sem serem somados aos que já estavam na tabela.

In [0]:
df_base_atendentes_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_base_atendentes")
df_base_motivos_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_base_motivos")
df_canais_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_canais")
df_chamados_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_chamados")
df_chamados_hora_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_chamados_hora")
df_clientes_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_clientes")
df_custos_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_custos")
df_pesquisa_satisfacao_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{bronze_schema}.tb_pesquisa_satisfacao")

## Validação da carga

Por fim, contamos as linhas de cada DataFrame para conferir se todos os arquivos foram lidos.

In [0]:
load_summary = [
    ("base_atendentes.csv", df_base_atendentes_raw.count()),
    ("base_motivos.csv", df_base_motivos_raw.count()),
    ("canais.csv", df_canais_raw.count()),
    ("chamados.csv", df_chamados_raw.count()),
    ("Chamados_Hora.CSV", df_chamados_hora_raw.count()),
    ("clientes.csv", df_clientes_raw.count()),
    ("custos.csv", df_custos_raw.count()),
    ("pesquisa_satisfacao.csv", df_pesquisa_satisfacao_raw.count()),
]

df_load_summary = spark.createDataFrame(load_summary, ["arquivo", "quantidade_registros"])
display(df_load_summary.orderBy("arquivo"))

arquivo,quantidade_registros
Chamados_Hora.CSV,131838
base_atendentes.csv,25
base_motivos.csv,14
canais.csv,4
chamados.csv,134474
clientes.csv,47705
custos.csv,131069
pesquisa_satisfacao.csv,81021
